# vector-normalize-keepdim — ex2: column-wise L2 normalize with keepdim=True (axis-flip of ex1)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `vector-normalize-keepdim`. Running the final beacon cell reports progress against the `PyTorch: vector normalize keepdim` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: vector normalize keepdim` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`vector-normalize-keepdim`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "vector-normalize-keepdim"
DD_SUBTOPIC = "PyTorch: vector normalize keepdim"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## row-wise vs column-wise normalize + the keepdim broadcast trap

Ex1 normalized rows with `dim=1, keepdim=True`. The deepening move is to swap to columns (`dim=0`) and show what `keepdim=False` would break.

```python
# Row-wise (ex1):    norms.shape == (N, 1) — broadcasts across columns.
# Column-wise (ex2): norms.shape == (1, M) — broadcasts across rows.
row_norms = x.norm(dim=1, keepdim=True)   # (N, 1)
col_norms = x.norm(dim=0, keepdim=True)   # (1, M)
```

**Why `keepdim=True` is load-bearing.** Without it, `x.norm(dim=0)` returns shape `(M,)`. Dividing a `(N, M)` tensor by a `(M,)` vector STILL broadcasts (right-aligned), so the column case happens to work without keepdim. But the row case (`dim=1` → `(N,)`) silently broadcasts WRONG: `(N, M) / (N,)` becomes `(N, M) / (1, N)`, which errors only if M != N — a sleeper bug on square matrices.

**`keepdim=True` is the safe habit.** The dropped axis is replaced by size 1, so broadcasting goes back to the axis you reduced over. Works for `dim=0` and `dim=1` identically — no row/column asymmetry.

### Exercise 2 — column-wise L2 normalize with keepdim=True (axis-flip of ex1)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `x.norm(dim=0, keepdim=True)` to L2-normalize the COLUMNS of a 2-D tensor — flipping ex1's row axis — while still leaning on keepdim to avoid the squeezed-shape broadcasting trap.
> Keywords: normalize, norm, keepdim, broadcast
> ```

**KCs targeted:** `norm-with-keepdim-preserves-rank`, `axis-flip-row-vs-column`

Implement `ex2_normalize_columns(x, eps=1e-12)`. The axis-flipped variant of ex1.

Inputs:
- `x`: `(N, M)` float tensor.
- `eps`: float, additive guard against divide-by-zero.

Algorithm:
1. `col_norms = x.norm(dim=0, keepdim=True)` — shape `(1, M)`.
2. Return `x / (col_norms + eps)`.

Constraints:
- DO NOT use `dim=1`. This drill is about the column axis specifically.
- DO NOT pass `keepdim=False`. Keep the reduced axis at size 1.
- Preserve the input dtype.
- DO NOT mutate `x`.

Output: `(N, M)` tensor where each COLUMN has L2 norm ≈ 1.0 (within `eps` of 1.0 for non-degenerate columns).

In [ ]:
def ex2_normalize_columns(x: Tensor, eps: float = 1e-12) -> Tensor:
    """L2-normalize each column. Uses x.norm(dim=0, keepdim=True)."""
    raise NotImplementedError()


def _test_ex2():
    # === Each column of the result has unit L2 norm ===
    t.manual_seed(0)
    x = t.randn(7, 4)
    y = ex2_normalize_columns(x)
    col_norms_out = y.norm(dim=0)  # (M,)
    assert t.allclose(col_norms_out, t.ones(4), atol=1e-5), (
        f'each column must have unit norm, got {col_norms_out}'
    )

    # === Shape + dtype preserved ===
    assert y.shape == x.shape, f'shape mismatch: in {tuple(x.shape)} vs out {tuple(y.shape)}'
    assert y.dtype == x.dtype, f'dtype must be preserved, got {y.dtype}'

    # === Direction preserved per column (cosine == 1) ===
    for c in range(x.shape[1]):
        cos = t.dot(x[:, c], y[:, c]) / (x[:, c].norm() * y[:, c].norm())
        assert t.allclose(cos, t.tensor(1.0), atol=1e-5), f'col {c} direction changed: cos={cos}'

    # === Input not mutated ===
    x_clone = x.clone()
    _ = ex2_normalize_columns(x_clone)
    assert t.equal(x_clone, x), 'must not mutate input'

    # === Hand-traced 3x2 ===
    x = t.tensor([
        [3.0, 0.0],
        [4.0, 0.0],
        [0.0, 5.0],
    ])
    # col 0 norm = 5, col 1 norm = 5 → result rows: (3/5,0), (4/5,0), (0,1)
    out = ex2_normalize_columns(x)
    expected = t.tensor([
        [0.6, 0.0],
        [0.8, 0.0],
        [0.0, 1.0],
    ])
    assert t.allclose(out, expected, atol=1e-5), f'expected={expected}, got {out}'

    # === Degenerate (zero) column → result is finite (eps guard) ===
    x = t.tensor([
        [0.0, 1.0],
        [0.0, 2.0],
        [0.0, 2.0],
    ])
    out = ex2_normalize_columns(x)
    assert t.isfinite(out).all(), f'eps guard failed; got {out}'
    # Zero column stays zero (0 / eps ≈ 0).
    assert t.allclose(out[:, 0], t.zeros(3), atol=1e-5)
    # Other column still has ~unit norm.
    assert t.isclose(out[:, 1].norm(), t.tensor(1.0), atol=1e-5)

    # === Square matrix (N == M) — keepdim is load-bearing here ===
    # If somebody used keepdim=False, the (M,) shape would broadcast as a row,
    # which on a square matrix would NOT error but would normalize the WRONG axis.
    # Use a non-symmetric value pattern to detect axis confusion.
    x = t.tensor([
        [1.0, 0.0, 0.0],
        [0.0, 2.0, 0.0],
        [0.0, 0.0, 3.0],
    ])
    out = ex2_normalize_columns(x)
    # Each column has only one non-zero. After normalize, each column is a one-hot.
    expected = t.eye(3)
    assert t.allclose(out, expected, atol=1e-5), (
        f'square-matrix axis confusion: expected eye(3), got {out}'
    )

    # === Single-row tensor → each column normalizes to ±1 ===
    x = t.tensor([[2.0, -3.0, 1.0]])
    out = ex2_normalize_columns(x)
    # Single-row L2 norm per column is |x[0, c]|, so each column becomes sign.
    expected = t.tensor([[1.0, -1.0, 1.0]])
    assert t.allclose(out, expected, atol=1e-5)

    # === Single-column tensor still works ===
    x = t.tensor([[3.0], [4.0]])
    out = ex2_normalize_columns(x)
    expected = t.tensor([[0.6], [0.8]])
    assert t.allclose(out, expected, atol=1e-5)
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_normalize_columns(x, eps=1e-12):
    col_norms = x.norm(dim=0, keepdim=True)  # (1, M)
    return x / (col_norms + eps)
```

**`dim=0` is the COLUMN reduce.** A common mental hiccup — `dim=0` SAVES axis 1 (columns); `dim=1` SAVES axis 0 (rows). The dim you pass is the dim that DISAPPEARS in the reduction.

**`keepdim=True` is non-optional for the square-matrix case.** On rectangular tensors, `(M,) / (N, M)` happens to broadcast correctly (right-aligned), but on square `(N, N)` it would silently normalize the wrong axis. The square-matrix test above catches that.

**`+ eps` over `clamp(min=eps)`.** Additive guard is one op; `clamp` is two. For inputs whose norm is genuinely zero, both give a near-zero output column, which is the right behavior — you can't recover a direction from the zero vector.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()